<a href="https://colab.research.google.com/github/DrMuratAltun/meb-yegitek-derin-ogrenme/blob/main/04_loss_fonksiyonlari.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a> &nbsp; <a href="https://github.com/DrMuratAltun/meb-yegitek-derin-ogrenme/blob/main/04_loss_fonksiyonlari.ipynb" target="_parent"><img src="https://img.shields.io/badge/GitHub-Kaynak-181717?logo=github" alt="GitHub'da Görüntüle"/></a>

> 🚀 **Bu notebook'u tarayıcıda çalıştırmak için sol üstteki "Colab'da Aç" butonuna tıklayın** — hiçbir şey kurmanıza gerek yok!


<div style="background: linear-gradient(135deg, #0B3D91 0%, #1565C0 60%, #1E88E5 100%); padding: 26px 30px; border-radius: 14px; color: white; margin: 0 0 24px 0; font-family: Georgia, serif; box-shadow: 0 4px 14px rgba(11,61,145,0.25);">

<div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px;">
  <span style="font-size: 11px; font-weight: bold; letter-spacing: 2px; color: #BBDEFB;">MEB · YEĞİTEK · DERİN ÖĞRENME SERİSİ</span>
  <span style="font-size: 11px; background: rgba(255,255,255,0.18); padding: 5px 12px; border-radius: 20px; color: white; font-weight: bold;">DERS 4 / 9 / 5</span>
</div>

<h1 style="margin: 8px 0 4px 0; color: white; font-size: 28px; line-height: 1.2;">📉 Loss Fonksiyonları — Hatayı Sayıyla Ölçmek</h1>

<p style="margin: 8px 0 0 0; color: #E3F2FD; font-size: 13px;">
<strong>Yenilik ve Eğitim Teknolojileri Genel Müdürlüğü</strong> — Öğretmenler için Uygulamalı Yapay Zeka<br>
<span style="color: #BBDEFB;">⏱️ Süre: ~40 dakika · 🖥️ Ortam: Google Colab veya yerel Python</span>
</p>

<div style="border-top: 1px solid rgba(255,255,255,0.22); margin-top: 14px; padding-top: 12px; font-size: 12px; color: #E3F2FD;">
🎯 Bu notebook <strong>non-teknik kitle</strong> için hazırlanmıştır · Kavramlar günlük hayattan analojilerle anlatılır · Kod minimum, açıklama maksimumdur.
</div>

</div>

## 🎯 Bu Notebook'ta Ne Öğreneceğiz?

İleri yayılımı öğrendik (notebook 03) — ağ artık tahmin yapabiliyor. Ama **tahmin doğru mu?** Doğru değilse **ne kadar yanlış?** Bunu **sayıyla** ölçmeliyiz ki ağırlıkları düzeltebilelim.

Bu işi yapan: **Kayıp Fonksiyonu (Loss Function)**.

Bu notebook'ta:

- 📐 **Kayıp** kavramı: Tahmin ne kadar uzakta?
- 📊 **MSE** (Mean Squared Error) — sayı tahmini için
- 🎯 **Binary Cross-Entropy** — 2 sınıf için
- 🌈 **Categorical Cross-Entropy** — N sınıf için
- 🤔 "Çok güvenli ama yanlış" tahmin neden ekstra cezalı?
- 🆚 NumPy'da elle yazıp Keras kıyas

## 🤔 Önce Bir Senaryo

Bir nişancı düşünün 🏹. Atışın iyi mi kötü mü olduğunu nasıl ölçeriz?

- **Hedefe ne kadar uzak düştü?** Mesafe → ham hata
- **Hedefe çok yakınsa az ceza, çok uzaksa çok ceza** → bunu nasıl yapsak?

İşte loss fonksiyonu **mesafeyi sayıya çeviren** matematik. Eğitim sürecinde model:

> "Bu turda hatam **0.42** çıktı. Bu sayıyı düşürmem lazım." der ve ağırlıklarını ona göre ayarlar.

Bu sayı **olmasaydı**, hangi yönde düzeleceğini bilmezdi. Loss = pusuladır.


## 📦 Kurulum

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print("✅ Hazır!")

## 1️⃣ MSE — Mean Squared Error (Ortalama Kare Hata)

**Ne zaman kullanılır?** Bir **sayı tahmini** yaparken (regresyon). Pizza fiyatı, ev fiyatı, hava sıcaklığı...

**Formül:**

$$\text{MSE} = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2$$

- $y_i$: gerçek değer
- $\hat{y}_i$: tahmin
- $N$: örnek sayısı

> 🎓 **Neden kare?** Eksiyi yer (hatanın yönü değil **büyüklüğü** önemli) ve **büyük hataları** orantısız fazla cezalandırır.


In [ ]:
def mse(y_gercek, y_tahmin):
    return np.mean((y_gercek - y_tahmin) ** 2)

# Pizza fiyatı senaryosu
gercek_fiyatlar  = np.array([100, 130, 150, 170])
iyi_tahminler    = np.array([105, 125, 148, 172])   # Yakın
kotu_tahminler   = np.array([ 80, 200, 100, 250])   # Uzak

print(f"İyi tahminler MSE  : {mse(gercek_fiyatlar, iyi_tahminler):.2f}")
print(f"Kötü tahminler MSE : {mse(gercek_fiyatlar, kotu_tahminler):.2f}")
print()
print("👀 Kötü tahminlerin MSE'si çok büyük çünkü kareleme uzaklığı büyütüyor.")

### MSE'nin Şeklini Görelim

Bir tahmin değeri için MSE eğrisi:

In [ ]:
# Gerçek değer 100. Tahmin 0'dan 200'e değişirse MSE eğrisi nasıl?
gercek = 100
tahminler = np.linspace(0, 200, 100)
mse_degerleri = (tahminler - gercek) ** 2

plt.plot(tahminler, mse_degerleri, color='#1E88E5', linewidth=3)
plt.axvline(x=gercek, color='green', linestyle='--', label=f'Gerçek değer: {gercek}')
plt.scatter([gercek], [0], color='green', s=200, zorder=5)
plt.xlabel('Tahmin'); plt.ylabel('MSE')
plt.title('📐 MSE Eğrisi — Tahmin gerçeğe yaklaştıkça hata 0\'a iner', fontweight='bold')
plt.legend(fontsize=11)
plt.show()

print("✅ MSE U şeklinde — minimum tam doğru tahminde.")
print("   Modelin amacı bu çukurun en altına inmek.")

## 2️⃣ Binary Cross-Entropy — 2 Sınıf İçin

**Ne zaman kullanılır?** Çıktının **iki sınıftan birine** ait olduğu durumda. Spam/spam değil, hasta/sağlıklı, kedi/köpek...

Modelin çıktısı **0-1 arası bir olasılık** (sigmoid'den geçer).

**Formül:**

$$\text{BCE} = -\frac{1}{N} \sum_{i=1}^{N} \big[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \big]$$

Karmaşık görünüyor ama mantığı basit:
- Gerçek 1'se: $-\log(\hat{y})$ — tahmin 1'e yakınsa hata az, 0'a yakınsa hata çok
- Gerçek 0'sa: $-\log(1 - \hat{y})$ — tahmin 0'a yakınsa hata az

> 🎓 **Logaritmanın sihri:** Tahmin tamamen yanlış (0.001 ama gerçek 1) ise hata **PATLAR**. Bu, model "çok güvenli ama yanlış" tahmin yapamaz.


In [ ]:
def binary_cross_entropy(y_gercek, y_tahmin):
    eps = 1e-15   # log(0) olmaması için minik ekleme
    y_tahmin = np.clip(y_tahmin, eps, 1 - eps)
    return -np.mean(y_gercek * np.log(y_tahmin) + (1 - y_gercek) * np.log(1 - y_tahmin))

# Spam tespiti senaryosu (1=spam, 0=spam değil)
gercek    = np.array([1, 0, 1, 0])
guvenli_dogru = np.array([0.95, 0.05, 0.92, 0.08])   # Çok iyi
guvensiz      = np.array([0.6, 0.4, 0.55, 0.45])     # Karasız
guvenli_yanlis= np.array([0.05, 0.95, 0.05, 0.95])   # KÖTÜ! Çok güvenli ama yanlış

print(f"Güvenli ve doğru   : BCE = {binary_cross_entropy(gercek, guvenli_dogru):.4f}")
print(f"Güvensiz tahminler  : BCE = {binary_cross_entropy(gercek, guvensiz):.4f}")
print(f"Güvenli ama yanlış  : BCE = {binary_cross_entropy(gercek, guvenli_yanlis):.4f}")
print()
print("👀 'Güvenli ama yanlış' modelin cezası ÇOK büyük — log fonksiyonu bunu sağlıyor!")

### BCE'nin Asimetrisini Görelim

In [ ]:
# Gerçek = 1 olduğunda, tahmin değişirse loss nasıl davranır?
tahminler = np.linspace(0.01, 0.99, 100)
bce_gercek_1 = -np.log(tahminler)            # Gerçek=1 ise -log(tahmin)
bce_gercek_0 = -np.log(1 - tahminler)        # Gerçek=0 ise -log(1-tahmin)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(tahminler, bce_gercek_1, color='#1E88E5', linewidth=3)
axes[0].set_title('Gerçek = 1', fontweight='bold')
axes[0].set_xlabel('Tahmin (olasılık)'); axes[0].set_ylabel('BCE')
axes[0].axvline(x=1, color='green', linestyle='--', alpha=0.5)

axes[1].plot(tahminler, bce_gercek_0, color='#E53935', linewidth=3)
axes[1].set_title('Gerçek = 0', fontweight='bold')
axes[1].set_xlabel('Tahmin (olasılık)'); axes[1].set_ylabel('BCE')
axes[1].axvline(x=0, color='green', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("✅ Tahmin yanlış sınıfa yaklaştıkça hata UÇAR. 'Güvenle yanlış olma' diyor.")

## 3️⃣ Categorical Cross-Entropy — N Sınıf İçin

**Ne zaman kullanılır?** İkiden fazla sınıf. Iris çiçekleri (3 sınıf), MNIST rakamları (10 sınıf), Fashion-MNIST giysileri (10 sınıf)...

Çıktı katmanı **softmax** kullanır — her sınıfa olasılık.

**Formül:**

$$\text{CCE} = -\sum_{i=1}^{C} y_i \log(\hat{y}_i)$$

Burada $y$ **one-hot** vektörü — sadece doğru sınıfta 1, diğerlerinde 0. Yani aslında **doğru sınıfın olasılığının log'u**:

$$\text{CCE} = -\log(\hat{y}_{\text{doğru sınıf}})$$

Çok basit! Doğru sınıfın olasılığı **1'e yakınsa** loss **küçük**, **0'a yakınsa** loss **büyük**.

In [ ]:
def categorical_cross_entropy(y_gercek_onehot, y_tahmin):
    eps = 1e-15
    y_tahmin = np.clip(y_tahmin, eps, 1 - eps)
    return -np.sum(y_gercek_onehot * np.log(y_tahmin), axis=1).mean()

# Iris senaryosu — 3 sınıf (Setosa, Versicolor, Virginica)
# Gerçek sınıflar (one-hot): 2 örnek var
y_gercek = np.array([
    [1, 0, 0],   # 1. örnek: Setosa
    [0, 0, 1],   # 2. örnek: Virginica
])

# Senaryolar
iyi    = np.array([[0.9, 0.05, 0.05], [0.1, 0.1, 0.8]])
karari_zor = np.array([[0.4, 0.3, 0.3],  [0.33, 0.33, 0.34]])
yanlis = np.array([[0.1, 0.7, 0.2],  [0.6, 0.3, 0.1]])

print(f"İyi tahmin       : CCE = {categorical_cross_entropy(y_gercek, iyi):.4f}")
print(f"Kararsız tahmin  : CCE = {categorical_cross_entropy(y_gercek, karari_zor):.4f}")
print(f"Yanlış tahmin    : CCE = {categorical_cross_entropy(y_gercek, yanlis):.4f}")

### Görsel: Doğru Sınıfın Olasılığı vs Loss

In [ ]:
olasiliklar = np.linspace(0.01, 1.0, 100)
loss = -np.log(olasiliklar)

plt.plot(olasiliklar, loss, color='#43A047', linewidth=3)
plt.xlabel('Doğru sınıfın tahmin olasılığı')
plt.ylabel('CCE (loss)')
plt.title('🎯 Doğru sınıfı %100 tahmin et → loss = 0', fontweight='bold')
plt.axvline(x=1.0, color='green', linestyle='--', alpha=0.5)
plt.show()

print("Olasılık 0.99 → loss ≈ 0.01 (mükemmel)")
print("Olasılık 0.50 → loss ≈ 0.69 (kararsız)")
print("Olasılık 0.10 → loss ≈ 2.30 (kötü)")
print("Olasılık 0.01 → loss ≈ 4.60 (felaket)")

## 🆚 Karma Demo: NumPy vs Keras

Bu üç loss'u Keras nasıl sunuyor?

In [ ]:
from tensorflow import keras

# Test verisi
y_true_reg = np.array([100., 130., 150.])
y_pred_reg = np.array([105., 125., 148.])

y_true_bin = np.array([1., 0., 1.])
y_pred_bin = np.array([0.9, 0.1, 0.85])

y_true_cat = np.array([[1,0,0], [0,0,1]], dtype='float32')
y_pred_cat = np.array([[0.9, 0.05, 0.05], [0.1, 0.1, 0.8]], dtype='float32')

print("                         NumPy             Keras")
print("-" * 60)
print(f"MSE                : {mse(y_true_reg, y_pred_reg):.4f}            "
      f"{keras.losses.MSE(y_true_reg, y_pred_reg).numpy():.4f}")
print(f"Binary Crossent    : {binary_cross_entropy(y_true_bin, y_pred_bin):.4f}            "
      f"{keras.losses.binary_crossentropy(y_true_bin, y_pred_bin).numpy():.4f}")
print(f"Categorical Cros.  : {categorical_cross_entropy(y_true_cat, y_pred_cat):.4f}            "
      f"{keras.losses.categorical_crossentropy(y_true_cat, y_pred_cat).numpy().mean():.4f}")
print()
print("👀 Tıpatıp aynı! Keras'taki `loss='mse'` yazınca tam bunu çağırıyor.")

## 🎮 Kendin Dene

Aşağıda gerçek değerleri ve tahminleri değiştirin, loss nasıl etkilenir görün:

In [ ]:
# 👇 Spam tespiti — kendi tahminlerinizi yazın (0-1 arası)
gercek_etiketler   = [1, 0, 1, 0, 1]      # Gerçek (1=spam, 0=spam değil)
tahmin_olasiliklari = [0.9, 0.2, 0.8, 0.1, 0.7]   # Modelinizin tahminleri

g = np.array(gercek_etiketler, dtype=float)
t = np.array(tahmin_olasiliklari, dtype=float)
loss = binary_cross_entropy(g, t)
dogru = (t > 0.5).astype(int)
acc = (dogru == g).mean()

print(f"BCE Loss  : {loss:.4f}")
print(f"Doğruluk  : {acc * 100:.0f}%")

## 📝 Özet

| Loss | Ne Zaman? | Çıktı Katmanı |
|---|---|---|
| **MSE** | Sayı tahmini (regresyon) | Lineer (aktivasyon yok) |
| **Binary Cross-Entropy** | 2 sınıf | Sigmoid |
| **Categorical Cross-Entropy** | 3+ sınıf, one-hot etiket | Softmax |
| **Sparse Cat. Cross-Ent.** | 3+ sınıf, **integer** etiket | Softmax |

> 💡 **Sparse vs Categorical:** Veri tek sayıyla (`2`) mı yoksa one-hot (`[0,0,1,0]`) mı? Tek sayıysa `sparse_categorical_crossentropy` kullan, **kendi one-hot'lamanı yapma**.

### 🎓 Anahtar İçgörü

> **Loss = Pusula. Modele "ne kadar yanlışsın, hangi yöne düzelt" der.**

Loss artık sayı olarak elimizde. Şimdi **bu sayıyı azaltmak için ağırlıkları nasıl güncelleyeceğiz?** İşte sıradaki notebook'un konusu: **Geri Yayılım** — sinir ağlarının kalbi! 💖

<div style="background: #0B3D91; color: #BBDEFB; padding: 26px 30px; border-radius: 14px; margin-top: 32px; font-family: Georgia, serif;">

<h3 style="color: white; margin: 0 0 12px 0; border-bottom: 2px solid #FFC107; padding-bottom: 8px; display: inline-block;">MEB · YEĞİTEK</h3>

<p style="font-family: Calibri, sans-serif; font-size: 14px; color: #E3F2FD; margin: 0 0 8px 0; line-height: 1.6;">
<strong>Yenilik ve Eğitim Teknolojileri Genel Müdürlüğü</strong><br>
Öğretmen ve eğitimciler için uygulamalı derin öğrenme eğitim serisi.
</p>

<p style="font-family: Calibri, sans-serif; font-size: 12px; color: #BBDEFB; margin: 12px 0 0 0;">
🎓 Bu notebook eğitim amaçlıdır · Kullanılan tüm kütüphaneler açık kaynaktır.
</p>

<div style="background: rgba(255,255,255,0.08); border-left: 3px solid #FFC107; padding: 12px 16px; margin-top: 16px; font-family: Calibri, sans-serif; font-size: 13px; color: #FFF9E6;">
➡️ <strong>Bir sonraki notebook:</strong> <code>05_geri_yayilim_optimizasyon.ipynb</code> — Geri yayılım ve optimizasyon — ağ nasıl öğrenir</div>
<p style="font-family: Calibri, sans-serif; font-size: 11px; color: #90CAF9; margin: 14px 0 0 0; text-align: center; border-top: 1px solid rgba(255,255,255,0.15); padding-top: 12px;">
© 2026 MEB YEĞİTEK · Derin Öğrenme Serisi
</p>

</div>